# Construir un scraper de X, paso a paso

Este cuaderno **no usa los programas del proyecto**. Construye un scraper
mínimo desde cero, celda por celda, para que veas exactamente de qué está hecho:
abrir el navegador, pedirle una búsqueda a X, leer una publicación de la
pantalla, repetir y guardar.

Al final tendrás un CSV con publicaciones reales y entenderás cada línea que lo
produjo. Son **24 pasos cortos**; ejecútalos en orden con `Shift+Enter`.

### Cómo funciona, en tres frases

X no regala sus datos, pero sí se los muestra a quien inició sesión. Así que el
scraper abre **Firefox de verdad**, sin ventana visible, reutilizando las
cookies de un perfil donde tú ya entraste a X. Desde ahí hace lo mismo que
harías tú: escribe en el buscador, baja por los resultados y lee lo que aparece.

No hay API de pago, ni contraseñas, ni nada escondido.

### Lo que vas a escribir

| Parte | Pasos | Qué construyes |
|---|---|---|
| 1 | 0–5 | Instalar, encontrar tu perfil de Firefox y abrir el navegador |
| 2 | 6–14 | Leer **una** publicación: autor, fecha, texto completo, favoritos |
| 3 | 15–17 | Recorrer la página hasta el final y guardar en CSV |
| 4 | 18–24 | Acotar por fechas, varias cuentas, limpiar, revisar y analizar |

> Al final, el paso 23 explica qué le falta a este scraper mínimo para servir en
> un corpus de verdad, y qué programas del proyecto se ocupan de eso.

---
# Parte 1 · Preparar

## Paso 0. Instalar

En la máquina necesitas dos cosas, que no se instalan desde aquí:

| Programa | Enlace |
|---|---|
| Python 3.11 o superior | <https://www.python.org/downloads/> |
| Firefox | <https://www.mozilla.org/firefox/new/> |

Si empiezas de cero, hay una guía pantalla por pantalla para
[Windows](INSTALACION_WINDOWS.md) y otra para [macOS](INSTALACION_MAC.md).

Las librerías de Python sí se instalan desde aquí. **Selenium** es la que maneja
el navegador; **pandas** es para mirar los datos al final.

In [ ]:
%pip install -q selenium pandas matplotlib

print("\nListo.")

## Paso 1. Comprobar que Selenium está

`selenium` es una librería que abre un navegador y lo maneja como si alguien
hiciera clic. Nada más.

In [ ]:
import sys
import selenium

print("Python  ", sys.version.split()[0])
print("selenium", selenium.__version__)

assert sys.version_info[:2] >= (3, 11), "Hace falta Python 3.11 o superior."

## Paso 2. Encontrar tu perfil de Firefox

Un *perfil* es la carpeta donde Firefox guarda pestañas, marcadores y
**cookies**. Las cookies son lo que nos interesa: ahí está la prueba de que
iniciaste sesión en X.

Firefox lleva un índice de sus perfiles en un archivo llamado `profiles.ini`.
Esta celda lo busca y los lista con la fecha de sus cookies.

**Antes de seguir**, si no lo has hecho:

1. Cierra Firefox del todo.
2. Ábrelo con el administrador de perfiles:
   - **Windows**: `Win+R` → `firefox.exe -P`
   - **macOS**: en la Terminal, `/Applications/Firefox.app/Contents/MacOS/firefox -P`
3. Crea uno llamado `scraper`, ábrelo y entra a <https://x.com> con tu cuenta.
4. **Cierra Firefox por completo.** Mientras esté abierto, el perfil queda
   bloqueado y no podremos usarlo.

In [ ]:
import configparser
import os
import time
from pathlib import Path


def perfiles_de_firefox():
    """Devuelve [(nombre, ruta, fecha de las cookies)] leyendo profiles.ini."""
    posibles = [
        Path.home() / "Library/Application Support/Firefox/profiles.ini",        # macOS
        Path.home() / ".mozilla/firefox/profiles.ini",                           # Linux
        Path.home() / "snap/firefox/common/.mozilla/firefox/profiles.ini",       # Linux (snap)
        Path(os.environ.get("APPDATA", "~")) / "Mozilla/Firefox/profiles.ini",   # Windows
    ]
    indice = next((ruta for ruta in posibles if ruta.is_file()), None)
    if indice is None:
        return []

    configuracion = configparser.ConfigParser()
    configuracion.read(indice, encoding="utf-8")
    encontrados = []
    for seccion in configuracion.sections():
        if not seccion.startswith("Profile"):
            continue
        ruta = Path(configuracion.get(seccion, "Path", fallback=""))
        if configuracion.getboolean(seccion, "IsRelative", fallback=True):
            ruta = indice.parent / ruta
        cookies = ruta / "cookies.sqlite"
        cuando = (time.strftime("%Y-%m-%d %H:%M", time.localtime(cookies.stat().st_mtime))
                  if cookies.is_file() else "sin cookies")
        encontrados.append((configuracion.get(seccion, "Name", fallback=seccion), ruta, cuando))
    return encontrados


for nombre, ruta, cuando in perfiles_de_firefox():
    print(f"{nombre:<22} cookies: {cuando}")
    print(f"   {ruta}")

print("\nEl de cookies más recientes suele ser donde acabas de iniciar sesión.")

## Paso 3. Elegir el perfil

Copia la ruta del perfil donde iniciaste sesión y pégala abajo **entre
comillas**. En Windows cambia las barras `\` por `/`.

Si lo dejas en `None`, tomamos el primero que tenga cookies. Funciona a veces,
pero es justo el origen del fallo más común: si el perfil elegido no tiene
sesión, X manda el navegador al formulario de acceso y **la descarga sale vacía
sin ningún mensaje de error**.

In [ ]:
PERFIL = None   # p. ej. "/Users/tu/Library/Application Support/Firefox/Profiles/a1b2c3d4.scraper"
                # Windows: "C:/Users/TU_USUARIO/AppData/Roaming/Mozilla/Firefox/Profiles/a1b2c3d4.scraper"

if PERFIL:
    perfil = Path(PERFIL).expanduser()
else:
    con_cookies = [ruta for _, ruta, cuando in perfiles_de_firefox() if cuando != "sin cookies"]
    perfil = con_cookies[0] if con_cookies else None

assert perfil and perfil.is_dir(), "No encontré el perfil. Escribe la ruta a mano arriba."
print("usaré este perfil:", perfil)

## Paso 4. Abrir el navegador

Aquí está el corazón de todo, y son ocho líneas.

`Options()` son las preferencias del navegador. Le pasamos el perfil —con eso
hereda tus cookies— y `--headless`, que significa *sin ventana*: Firefox corre
igual, pero no lo ves. Si quieres verlo trabajar, llama a la función con
`visible=True`.

In [ ]:
from selenium import webdriver
from selenium.common.exceptions import StaleElementReferenceException
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.firefox_profile import FirefoxProfile
from selenium.webdriver.firefox.options import Options


def abrir_navegador(perfil, visible=False):
    """Abre Firefox con las cookies de ese perfil."""
    opciones = Options()
    opciones.profile = FirefoxProfile(str(perfil))     # <- aquí hereda la sesión
    if not visible:
        opciones.add_argument("--headless")
    opciones.set_preference("intl.accept_languages", "es-CO,es,es-419")
    navegador = webdriver.Firefox(options=opciones)
    navegador.set_window_size(1920, 1080)
    return navegador


navegador = abrir_navegador(perfil)
print("navegador abierto")

> **Si esta celda falló** diciendo algo de `geckodriver`: es el puente entre
> Python y Firefox. Selenium lo descarga solo, pero algunas redes
> institucionales lo bloquean. Las guías de instalación explican cómo bajarlo a
> mano.
>
> **Si dijo que el perfil está en uso**: cierra Firefox del todo y repite.

El navegador queda abierto durante todo el cuaderno. **La última celda lo
cierra**; si interrumpes a mitad de camino, ejecútala para no dejar un Firefox
invisible corriendo.

## Paso 5. Probar la sesión

Vamos a la portada de X y miramos quién cree que somos. Si la sesión sirve, no
aparece el formulario de acceso.

In [ ]:
navegador.get("https://x.com/home")
time.sleep(6)

hay_formulario = bool(navegador.find_elements(By.CSS_SELECTOR, 'input[autocomplete="username"]'))
en_login = "/i/flow/login" in navegador.current_url

print("dirección actual:", navegador.current_url)
print("¿pide iniciar sesión?:", "SÍ" if (hay_formulario or en_login) else "no")

assert not (hay_formulario or en_login), (
    "Ese perfil no tiene sesión en X. Vuelve al paso 2, entra a x.com a mano "
    "en ese perfil y cierra Firefox antes de repetir."
)
print("\nSesión activa. Podemos buscar.")

---
# Parte 2 · Leer una publicación

## Paso 6. Armar la dirección de búsqueda

Cuando buscas en X, el navegador va a una dirección como
`https://x.com/search?q=...&f=live`. Eso es todo lo que vamos a construir.

Los operadores útiles:

| Operador | Significa |
|---|---|
| `from:cuenta` | Sólo de esa cuenta |
| `since:` / `until:` | Desde / hasta. `until:` **no incluye** ese instante |
| `"dos palabras"` | La frase exacta |
| `(a OR b)` | Cualquiera de las dos |
| `-filter:replies` | Sin respuestas |

Y `f=live` pide la pestaña **Más recientes** en vez de **Destacados**: nos
interesa todo, no lo que X considere popular.

In [ ]:
from urllib.parse import urlencode


def url_de_busqueda(consulta):
    """Convierte una consulta de X en la dirección del buscador."""
    return "https://x.com/search?" + urlencode({
        "q": consulta,
        "src": "typed_query",
        "f": "live",          # "Más recientes", no "Destacados"
    })


print(url_de_busqueda("from:IvanCepedaCast"))
print(url_de_busqueda('(#PrioridadNacional OR "Prioridad Nacional")'))

## Paso 7. Cargar una búsqueda y contar lo que hay

Cada publicación en pantalla es un elemento HTML `<article>` que X marca con
`data-testid="tweet"`. Ese es nuestro anzuelo.

Pedimos una cuenta activa y contamos cuántos artículos cargaron.

In [ ]:
ARTICULO = 'article[data-testid="tweet"]'

navegador.get(url_de_busqueda("from:IvanCepedaCast"))
time.sleep(6)

articulos = navegador.find_elements(By.CSS_SELECTOR, ARTICULO)
print("publicaciones visibles ahora mismo:", len(articulos))
print("\nX carga unas pocas y va añadiendo más a medida que uno baja.")

## Paso 8. De un artículo, sacar el autor y el identificador

Dentro de cada artículo hay un enlace a la publicación, con esta forma:

```
https://x.com/IvanCepedaCast/status/1234567890123456789
```

Ahí están las dos cosas que necesitamos: el **usuario** y el **identificador**.
Ese número es único e irrepetible: es la clave con la que después evitaremos
duplicados. Lo sacamos con una expresión regular.

Fíjate en el `try`: X redibuja la página todo el tiempo, y una publicación que
tenías localizada puede desaparecer entre una línea y la siguiente. Devolver
`None` en vez de reventar es lo correcto; en el paso 12 se ve por qué.

In [ ]:
import re

PATRON_ENLACE = re.compile(r"(?:x|twitter)\.com/([A-Za-z0-9_]{1,15})/status/(\d+)")


def identidad(articulo):
    """Devuelve (usuario, id, url) leyendo el primer enlace /status/ del artículo.

    Si el artículo ya no está en la página devuelve (None, None, None) en vez de
    reventar: X redibuja constantemente y eso es normal, no un error.
    """
    try:
        enlaces = articulo.find_elements(By.CSS_SELECTOR, 'a[href*="/status/"]')
    except StaleElementReferenceException:
        return None, None, None

    for enlace in enlaces:
        try:
            destino = enlace.get_attribute("href") or ""
        except StaleElementReferenceException:
            continue
        encontrado = PATRON_ENLACE.search(destino)
        if encontrado:
            usuario, identificador = encontrado.groups()
            return usuario, identificador, f"https://x.com/{usuario}/status/{identificador}"
    return None, None, None


primero = articulos[0]
print(identidad(primero))

## Paso 9. Sacar la fecha

X pone la fecha en un elemento `<time>`, con el momento exacto en el atributo
`datetime`, en UTC y en formato ISO. Es mucho más fiable que el texto que se ve
("hace 2 h"), que depende de cuándo mires.

In [ ]:
def fecha(articulo):
    """Momento de publicación, en UTC e ISO. Ej: 2026-06-20T14:03:11.000Z"""
    elementos = articulo.find_elements(By.TAG_NAME, "time")
    return elementos[0].get_attribute("datetime") if elementos else None


print("fecha:", fecha(primero))
print("lo que ve el lector:", primero.find_elements(By.TAG_NAME, "time")[0].text)

## Paso 10. Sacar el texto… y descubrir el problema

El texto vive en un `div` marcado como `tweetText`. Parece fácil.

Pero aquí está la trampa que arruina corpus enteros: **en los resultados de
búsqueda, X recorta las publicaciones largas a unos 280 caracteres y no deja
ninguna marca**. No hay puntos suspensivos, la frase simplemente se interrumpe.
Si guardas eso, tienes un archivo que parece perfecto y está mutilado.

La celda siguiente lee los textos tal cual y mira cuántos quedan sospechosamente
cerca de 280.

In [ ]:
def texto_crudo(articulo):
    try:
        partes = articulo.find_elements(By.CSS_SELECTOR, 'div[data-testid="tweetText"]')
        return partes[0].text.strip() if partes else ""
    except StaleElementReferenceException:
        return ""


longitudes = []
for articulo in articulos:
    contenido = texto_crudo(articulo)
    if contenido:
        longitudes.append(len(contenido))

print("longitudes de los textos en pantalla:")
print(sorted(longitudes))
print()
print("cerca de 280 (entre 270 y 285):", sum(1 for n in longitudes if 270 <= n <= 285))

## Paso 11. Abrir los textos largos

Cuando X recorta, deja un botón **"Mostrar más"**. Hay que pulsarlo antes de
leer.

Lo pulsamos con JavaScript (`execute_script`) en vez de con un clic normal,
porque el botón puede estar tapado por otra cosa y un clic normal fallaría.

Ojo con no confundirlo con "Mostrar esta cadena", que abre el hilo completo: por
eso buscamos el marcador exacto `tweet-text-show-more-link`.

In [ ]:
def abrir_texto_largo(navegador, articulo):
    """Pulsa 'Mostrar más' si la publicación está recortada."""
    try:
        botones = articulo.find_elements(
            By.CSS_SELECTOR, '[data-testid="tweet-text-show-more-link"]'
        )
        if not botones:
            botones = articulo.find_elements(
                By.XPATH,
                './/button[normalize-space()="Mostrar más" or normalize-space()="Show more"]'
                ' | .//span[normalize-space()="Mostrar más" or normalize-space()="Show more"]',
            )
        for boton in botones:
            navegador.execute_script("arguments[0].click();", boton)
        return bool(botones)
    except StaleElementReferenceException:
        return False


# Buscamos una publicación recortada y comparamos antes y después.
for articulo in articulos:
    antes = texto_crudo(articulo)
    if abrir_texto_largo(navegador, articulo):
        time.sleep(1)
        despues = texto_crudo(articulo)
        print(f"antes:   {len(antes)} caracteres -> ...{antes[-60:]!r}")
        print(f"después: {len(despues)} caracteres -> ...{despues[-60:]!r}")
        break
else:
    print("Ninguna de las publicaciones en pantalla estaba recortada.")
    print("Pasa igual: la función se usa en todas por si acaso.")

## Paso 12. Deshacer la traducción

Otra trampa, y de las caras. **X traduce las publicaciones** cuando cree que
están en otro idioma, y lo que queda en pantalla es la traducción automática.
Si la lees tal cual, tu corpus en español termina lleno de inglés de Google.

No es hipotético: la primera versión de este cuaderno guardó **96 de 115
publicaciones traducidas** sin que nada lo advirtiera.

Cuando X traduce, deja un enlace **"Ver original"**. Hay que pulsarlo antes de
leer.

> **Aquí aparece el error clásico de Selenium.** Al pulsar, X vuelve a dibujar
> la página, y todas las referencias a elementos que tuvieras guardadas dejan de
> servir: Selenium las llama *stale* y lanza `StaleElementReferenceException`.
>
> Por eso de aquí en adelante **no guardamos artículos, guardamos
> identificadores**, y volvemos a buscar el artículo justo antes de usarlo. Es
> la diferencia entre anotar la dirección de una casa y anotar dónde estaba
> parado el cartero.

In [ ]:
def ver_original(navegador, articulo):
    """Si X tradujo la publicación, vuelve al texto original."""
    try:
        enlaces = articulo.find_elements(
            By.XPATH,
            './/span[contains(normalize-space(), "Ver original") or '
            'contains(normalize-space(), "Show original") or '
            'contains(normalize-space(), "Ver texto original")]',
        )
        for enlace in enlaces:
            navegador.execute_script("arguments[0].click();", enlace)
            time.sleep(0.8)
        return bool(enlaces)
    except StaleElementReferenceException:
        return False


def buscar_articulo(navegador, identificador):
    """Vuelve a localizar un artículo por su id, después de que X lo redibuje."""
    for articulo in navegador.find_elements(By.CSS_SELECTOR, ARTICULO):
        _, otro, _ = identidad(articulo)      # identidad() ya tolera los caducados
        if otro == identificador:
            return articulo
    return None


# Primero anotamos los identificadores; después trabajamos de uno en uno.
identificadores = []
for articulo in navegador.find_elements(By.CSS_SELECTOR, ARTICULO):
    try:
        _, identificador, _ = identidad(articulo)
    except StaleElementReferenceException:
        continue
    if identificador:
        identificadores.append(identificador)

traducidas = 0
for identificador in identificadores:
    articulo = buscar_articulo(navegador, identificador)
    if articulo is None:
        continue
    antes = texto_crudo(articulo)
    if ver_original(navegador, articulo):
        traducidas += 1
        vuelto = buscar_articulo(navegador, identificador)   # <- buscarlo otra vez
        if vuelto is not None and traducidas == 1:
            print("traducción:", antes[:70])
            print("original  :", texto_crudo(vuelto)[:70])

print(f"\nde {len(identificadores)} publicaciones, X había traducido {traducidas}")
if traducidas == 0:
    print("En esta carga X no tradujo nada. Pasa: depende del idioma de tu cuenta")
    print("y del momento. La comprobación se hace igual en todas, por si acaso.")

## Paso 13. Sacar los favoritos

El botón de "me gusta" lleva el número. X lo muestra abreviado (`2.1K`), así que
guardamos las dos versiones: la original, por si quieres auditar, y el número
para poder sumar.

In [ ]:
def numero(texto):
    """Convierte '2.1K', '10 mil', '1,234' o '1.234' en un entero.

    X abrevia distinto según el idioma, y la coma es separador de miles en
    inglés y decimal en español: hay que mirar el sufijo antes de quitar nada.
    """
    if not texto:
        return None

    limpio = texto.strip().upper().replace(chr(160), " ")   # chr(160): el espacio duro que usa X
    con_sufijo = re.search(r"(\d+(?:[.,]\d+)?)\s*(K|MIL|M)", limpio)
    if con_sufijo:
        cantidad = float(con_sufijo.group(1).replace(",", "."))
        factor = {"K": 1_000, "MIL": 1_000, "M": 1_000_000}[con_sufijo.group(2)]
        return round(cantidad * factor)

    solo_digitos = re.sub(r"\D", "", limpio)
    return int(solo_digitos) if solo_digitos else None


def favoritos(articulo):
    botones = articulo.find_elements(
        By.CSS_SELECTOR, 'button[data-testid="like"], button[data-testid="unlike"]'
    )
    if not botones:
        return "", None
    crudo = botones[0].text.strip()
    return crudo, numero(crudo)


for articulo in articulos[:5]:
    print(favoritos(articulo))

## Paso 14. Juntarlo todo

Una función que recibe un **identificador** y devuelve un diccionario con la
publicación completa. Es la suma de los pasos anteriores.

Recibe el identificador y no el artículo, por lo que vimos en el paso 12: entre
que lo encontramos y que lo leemos, X puede haber redibujado la página.

Fíjate en el orden, que no es casual: deshacer la traducción, abrir el texto
largo, y sólo entonces leer. Al revés guardaríamos una traducción recortada.

In [ ]:
from datetime import datetime, timezone


def leer_publicacion(navegador, identificador, articulo=None):
    """Convierte una publicación en un diccionario, buscándola por su id.

    Se le puede pasar el artículo ya localizado para ahorrar un barrido; si
    resulta estar caducado, lo busca de nuevo.
    """
    if articulo is None:
        articulo = buscar_articulo(navegador, identificador)
    if articulo is None:
        return None

    usuario, hallado, enlace = identidad(articulo)
    if hallado != identificador:          # caducó entre que lo vimos y lo leímos
        articulo = buscar_articulo(navegador, identificador)
        if articulo is None:
            return None
        usuario, _, enlace = identidad(articulo)

    if ver_original(navegador, articulo):           # 1. deshacer la traducción
        articulo = buscar_articulo(navegador, identificador)
        if articulo is None:
            return None
    abrir_texto_largo(navegador, articulo)          # 2. abrir el texto largo
    contenido = texto_crudo(articulo)               # 3. y sólo entonces leer
    crudo, favs = favoritos(articulo)

    return {
        "tweet_id": identificador,
        "username": usuario,
        "date": fecha(articulo),
        "text": contenido,
        "favorites_raw": crudo,
        "favorites": favs,
        "url": enlace,
        "scraped_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }


ejemplo = leer_publicacion(navegador, identificadores[0])
for clave, valor in ejemplo.items():
    print(f"{clave:>13}: {str(valor)[:90]}")

---
# Parte 3 · Recorrer y guardar

## Paso 15. Bajar para que carguen más

X no muestra todo de una vez: va añadiendo publicaciones a medida que bajas. Eso
se llama *scroll infinito*, y para el scraper significa que hay que bajar,
esperar a que carguen y volver a mirar.

También descarta las de arriba cuando te alejas mucho, así que **hay que leerlas
a medida que aparecen**, no al final.

In [ ]:
antes = len(navegador.find_elements(By.CSS_SELECTOR, ARTICULO))

navegador.execute_script("window.scrollBy(0, window.innerHeight * 2);")
time.sleep(3)

despues = len(navegador.find_elements(By.CSS_SELECTOR, ARTICULO))
print(f"artículos en pantalla: {antes} -> {despues}")

## Paso 16. El bucle completo

Ya está todo. El scraper es: cargar la búsqueda, leer lo visible, bajar, repetir
hasta que no aparezca nada nuevo.

Dos detalles que importan:

- Va en **dos fases**: primero anota qué identificadores hay en pantalla, y
  después los lee uno a uno buscando cada artículo de nuevo. Es lo que evita el
  `StaleElementReferenceException` del paso 12.
- Guardamos en un **diccionario con el identificador como clave**: así, si una
  publicación se lee dos veces, se queda una sola.
- Paramos cuando **tres bajadas seguidas** no traen nada nuevo. Una sola no
  basta: a veces X tarda en responder y parecería que se acabó.

In [ ]:
def recolectar(navegador, consulta, max_bajadas=15, espera=3.0):
    """Recorre los resultados de una búsqueda y devuelve las publicaciones."""
    navegador.get(url_de_busqueda(consulta))
    time.sleep(6)

    encontradas = {}
    sin_novedad = 0

    for vuelta in range(max_bajadas):
        antes = len(encontradas)

        # Fase 1: anotar qué hay en pantalla, sin tocar nada.
        pendientes = []
        for articulo in navegador.find_elements(By.CSS_SELECTOR, ARTICULO):
            _, identificador, _ = identidad(articulo)
            if identificador and identificador not in encontradas:
                pendientes.append((identificador, articulo))

        # Fase 2: leerlas una a una. Si alguna caducó, leer_publicacion la busca.
        for identificador, articulo in pendientes:
            try:
                registro = leer_publicacion(navegador, identificador, articulo)
            except StaleElementReferenceException:
                continue        # X la quitó de la pantalla; aparecerá en otra vuelta
            if registro:
                encontradas[identificador] = registro

        nuevas = len(encontradas) - antes
        print(f"  vuelta {vuelta + 1}: +{nuevas} nuevas, {len(encontradas)} en total")

        sin_novedad = sin_novedad + 1 if nuevas == 0 else 0
        if sin_novedad >= 3:
            print("  tres vueltas sin nada nuevo: se acabó")
            break

        navegador.execute_script("window.scrollBy(0, window.innerHeight * 2);")
        time.sleep(espera)

    return list(encontradas.values())


publicaciones = recolectar(navegador, "from:IvanCepedaCast")
print("\ntotal:", len(publicaciones))

## Paso 17. Guardar en CSV

Con pandas. Dos detalles para que Excel en español lo abra bien: separador `;` y
codificación `utf-8-sig`.

> **Nunca mires este archivo con `head`, `tail` o `wc -l`.** Los textos
> contienen saltos de línea —así escribe la gente—, de modo que una fila ocupa
> varias líneas del archivo y esas herramientas la parten a la mitad. Ábrelo
> siempre con un lector de CSV.

In [ ]:
import pandas as pd

tabla = pd.DataFrame(publicaciones)
tabla["date"] = pd.to_datetime(tabla["date"], format="ISO8601", utc=True)
tabla = tabla.sort_values("date", ascending=False)

tabla.to_csv("mi_corpus.csv", sep=";", index=False, encoding="utf-8-sig")
print("guardadas", len(tabla), "publicaciones en mi_corpus.csv\n")

tabla[["date", "username", "favorites", "text"]].head(3)

---
# Parte 4 · Hacerlo bien

## Paso 18. Acotar por fechas

Hasta aquí bajamos "lo último". Para un estudio hace falta un periodo exacto, y
ahí hay una trampa.

Si escribes `since:2026-06-01`, X interpreta esa fecha **en su propio huso
horario**, no en el tuyo, y el borde del periodo se te mueve sin avisar. La
solución es dar la hora y el huso completos:

```
since:2026-06-01_00:00:00_UTC until:2026-07-01_00:00:00_UTC
```

`until:` **no incluye** ese instante: lo de arriba cubre todo junio.

In [ ]:
def consulta_con_fechas(cuenta, desde, hasta):
    """desde/hasta en formato AAAA-MM-DD, interpretados como UTC."""
    return (f"from:{cuenta}"
            f" since:{desde}_00:00:00_UTC"
            f" until:{hasta}_00:00:00_UTC")


DESDE, HASTA = "2026-06-01", "2026-06-08"

consulta = consulta_con_fechas("IvanCepedaCast", DESDE, HASTA)
print(consulta)
print()

semana = recolectar(navegador, consulta)
print("\npublicaciones de esa semana:", len(semana))

## Paso 19. Varias cuentas

Un estudio normalmente son varias personas. El bucle es el obvio: una búsqueda
por cuenta, y todo a la misma lista.

> **`max_bajadas=8` recorta a propósito.** Con ocho vueltas no se baja todo lo
> que publicó cada cuenta esa semana: es una demostración, no un corpus. Para
> bajar de verdad, sube el límite y deja que pare solo cuando se acaben los
> resultados.
>
> **Esta celda tarda.** Varios minutos: cada publicación implica
> varias idas y vueltas al navegador, y X responde cada vez más despacio a
> medida que avanza la sesión. No se ha colgado.
>
> **X limita.** Aguanta unas **12 búsquedas nuevas** seguidas y después bloquea
> entre 9 y 15 minutos. Por eso aquí sólo van tres cuentas y una semana. Si ves
> que empiezan a salir cero publicaciones de cuentas que sí publican, no es tu
> código: es el bloqueo.

In [ ]:
CUENTAS = ["IvanCepedaCast", "MafeCarrascal", "AlfreMondragon"]

todas = []
for cuenta in CUENTAS:
    print(f"\n--- {cuenta} ---")
    consulta = consulta_con_fechas(cuenta, DESDE, HASTA)
    encontradas = recolectar(navegador, consulta, max_bajadas=8)
    for registro in encontradas:
        registro["cuenta_pedida"] = cuenta      # para saber a quién se pidió
    todas.extend(encontradas)
    time.sleep(5)                               # un respiro entre búsquedas

corpus = pd.DataFrame(todas)
corpus["date"] = pd.to_datetime(corpus["date"], format="ISO8601", utc=True)

print("\ntotal descargado:", len(corpus))
print(corpus["cuenta_pedida"].value_counts().to_string())

## Paso 20. Quedarte con lo que pediste

Mira la tabla de arriba: aparecen cuentas que **no** están en tu lista.

No es un error del código. Cuando una publicación es respuesta a otra, X dibuja
también la publicación original encima, y para el navegador son dos artículos
iguales. El scraper, que lee todo lo que encuentra, se los trae.

En la prueba que hicimos al escribir este cuaderno, **43 de 115 publicaciones
eran de otra persona**, había 3 repetidas y una de enero del año anterior. Si
analizas eso sin mirar, estás contando publicaciones ajenas como propias.

La solución no es tocar el scraper: es filtrar al final, con las tres
condiciones que de verdad definen lo que pediste.

In [ ]:
antes_de_limpiar = len(corpus)

del_autor = corpus["username"].str.lower() == corpus["cuenta_pedida"].str.lower()
en_rango = corpus["date"].between(
    pd.Timestamp(DESDE, tz="UTC"), pd.Timestamp(HASTA, tz="UTC"), inclusive="left"
)

corpus_limpio = corpus[del_autor & en_rango].drop_duplicates("tweet_id").copy()

print(f"descargadas          : {antes_de_limpiar}")
print(f"de otra cuenta       : {int((~del_autor).sum())}  (contexto de respuestas)")
print(f"fuera del periodo    : {int((~en_rango).sum())}")
print(f"repetidas            : {int(corpus['tweet_id'].duplicated().sum())}")
print(f"se quedan            : {len(corpus_limpio)}")

corpus_limpio.to_csv("mi_corpus.csv", sep=";", index=False, encoding="utf-8-sig")

## Paso 21. Revisar antes de analizar

Tres comprobaciones. Hazlas siempre, antes de sacar cualquier conclusión.

La tercera es la menos evidente: si muchísimos textos miden entre 270 y 285
caracteres y casi ninguno pasa de 300, es que están cortados en los 280. La
gente no escribe así de parejo.

Sobre los **textos vacíos**: casi siempre son publicaciones de sólo imagen o
vídeo, que existen y cuentan. Si las descartas o no, decídelo tú y déjalo
escrito; lo que no vale es que desaparezcan sin que nadie lo sepa.

In [ ]:
largo = corpus_limpio["text"].str.len()
banda = int(largo.between(270, 285).sum())
largas = int((largo > 300).sum())

print("publicaciones      :", len(corpus_limpio))
print("ids duplicados     :", int(corpus_limpio["tweet_id"].duplicated().sum()))
vacios = int(corpus_limpio["text"].str.strip().eq("").sum())
print("textos vacíos      :", vacios, "(normalmente publicaciones de sólo imagen o vídeo)")

distinto = (corpus_limpio["username"].str.lower() != corpus_limpio["cuenta_pedida"].str.lower()).sum()
print("autor != pedida    :", int(distinto), "(debe ser 0)")

print()
print("longitud media     : %.0f" % largo.mean())
print("entre 270 y 285    : %d (%.1f%%)" % (banda, 100 * banda / len(corpus_limpio)))
print("más de 300         : %d" % largas)
print()

if len(corpus_limpio) < 300:
    print("Muestra pequeña: la comprobación de textos cortados necesita unos")
    print("cientos de publicaciones para ser fiable. Aquí es sólo ilustrativa.")
elif 100 * banda / len(corpus_limpio) > 20 and banda > 3 * largas:
    print("SOSPECHA de textos cortados: revisa el paso 11.")
else:
    print("Sin señales de truncamiento sistemático.")

## Paso 22. Un primer vistazo

In [ ]:
print("publicaciones por día:")
print(corpus_limpio.set_index("date").resample("D").size().to_string())

print("\nlas de más favoritos:\n")
columnas = ["date", "username", "favorites", "text"]
print(corpus_limpio.nlargest(3, "favorites")[columnas].to_string(index=False, max_colwidth=60))

In [ ]:
try:
    import matplotlib.pyplot as plt

    conteo = corpus_limpio.set_index("date").resample("D").size()
    eje = conteo.plot(kind="bar", figsize=(9, 3), title="Publicaciones por día")
    eje.set_xlabel("")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Instala matplotlib para ver el gráfico.")

## Paso 23. Qué le falta a esto para un corpus de verdad

Lo que escribiste funciona y es honesto: para una semana de tres cuentas, sirve.
Para cuatro años de cincuenta cuentas, no, y conviene saber por qué.

| Problema | Qué pasa | Qué haría falta |
|---|---|---|
| **Se pierde todo si se corta** | Un bloqueo de X a las seis horas y vuelves a empezar | Guardar cada tramo terminado y poder continuar |
| **X corta la paginación sin avisar** | Devuelve 40 publicaciones donde había 300, sin error | Comparar con el ritmo de publicación y volver a pedir el tramo partido en dos |
| **Los bloqueos** | `time.sleep(5)` no alcanza | Esperas crecientes y reintentos |
| **Una ventana muy larga** | X nunca entrega años de un tirón | Partir en lotes de un trimestre |

Eso es exactamente lo que hacen los programas del proyecto, que son este mismo
código más esos cuidados:

```
python descargar_cuentas.py --roster censo.json --window-months 3
python descargar_busqueda.py --term "#PrioridadNacional" --start 2026-06-01 --end 2026-07-01
```

El `README.md` los explica. Con ellos se bajó el corpus de ejemplo: 54
congresistas, cuatro años, 212.962 publicaciones, varios días de descarga.

Lo importante es que ahora sabes qué hacen por dentro, porque acabas de
escribirlo.

## Paso 24. Cerrar el navegador

**Ejecuta siempre esta celda al terminar.** Si no, queda un Firefox invisible
consumiendo memoria. Si interrumpiste el cuaderno a medias, ejecútala también.

In [ ]:
navegador.quit()
print("navegador cerrado")